# grid_100x100 PIE パトロール（A* + SpotDog 往復）

**前提**: UE Editor で `grid_100x100` を開き **PIE 実行中** にこのノートを実行してください。

## シナリオ

1. 外周ブロックを実体化 (`SetBlocking True`)
2. Humanoid をマス **(5, 5)**、SpotDog を **(10, 5)** にスポーン
3. SpotDog が A* 経路で **(50, 50)** へ移動 → **5 秒停止**
4. 出発点 **(10, 5)** へ A* で帰還

ロジック: `grid_env_10k_pie_patrol.py`（コストマップ解像度 0.30 m = ブロック幅、`path_planning_costmap` の格子 A*）

カーネル: `conda activate simworld`

## 初回セットアップ（Editor PIE）

1. **Robot_Dog** を Editor プロジェクトへコピー（未導入時）:
   `bash dev/grid_env_10k/scripts/install_robot_dog_editor.sh`
2. UE Editor を再起動するか、**PIE を Stop → Play** してアセットを再読込
3. Outliner の `block_*` はラベル名。UnrealCV は `BP_TransparentCube_C_UAID_...` を使うため、本スクリプトが位置から自動解決します

## 同じ PIE セッションで再実行するとき

- **再実行は Cell 5 のみ**（パラメータは `patrol` の既定値。変更するときだけ Cell 4 を先に実行）
- `ensure_connection()` は **既存の UnrealCV TCP クライアントを再利用**（2 本目の接続は開かない）
- Windows の `Get-NetTCPConnection -LocalPort 9000` で **Listen（UE）+ Established（Python 1 本）** は正常
- `skip_perimeter_if_already_solid=True` … 外周 T 化をスキップ（約8分短縮）
- `skip_robot_probe=True` … BP プローブをスキップ（Rename クラッシュ回避）
- 前回の SpotDog が残っている場合は **再スポーンせずテレポート** で起点に戻します
- **毎回 PIE 再起動は不要**。接続だけ壊れたとき: Cell 3 の `release_ue_connection()` → Cell 2 → Cell 5
- クラッシュした場合は PIE Stop → Play してから Cell 2 から再実行

In [1]:
import sys
print(sys.executable)

/home/winder17wsl_ishizawalab/miniforge3/envs/simworld/bin/python


In [2]:
import importlib
from pathlib import Path
from typing import Optional, Tuple

from simworld.communicator.communicator import Communicator
from simworld.communicator.unrealcv import UnrealCV


def _find_project_root() -> Path:
    for start in (Path.cwd().resolve(), Path(".").resolve()):
        for candidate in (start, *start.parents):
            if (candidate / "setup.py").exists() and (candidate / "simworld").is_dir():
                return candidate
    return Path.cwd().resolve().parent.parent


_root = _find_project_root()
_geh_dir = _root / "dev" / "grid_env_hri"
_g10k = _root / "dev" / "grid_env_10k"
for p in (_root, _geh_dir, _g10k):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Keep live TCP client across importlib.reload (UnrealCV is single-client).
_prev_ucv: Optional[UnrealCV] = globals().get("ucv")
_prev_comm: Optional[Communicator] = globals().get("communicator")

import grid_env_hri_simulation as geh
import grid_env_10k as g10k
import grid_env_10k_pie_patrol as patrol

importlib.reload(geh)
importlib.reload(g10k)
importlib.reload(patrol)

ucv: Optional[UnrealCV] = None
communicator: Optional[Communicator] = None

if _prev_ucv is not None:
    try:
        ucv, communicator = g10k.adopt_ue_session(_prev_ucv, _prev_comm)
    except (ConnectionError, Exception):
        try:
            _prev_ucv.disconnect()
        except Exception:
            pass
        print("[UE] prior session not reusable — will connect on ensure_connection()")


def ensure_connection() -> Tuple[UnrealCV, Communicator]:
    """Reuse one UnrealCV TCP session across Cell 5 re-runs (no second client on :9000)."""
    global ucv, communicator
    ucv, communicator = g10k.ensure_connection(ucv=ucv, communicator=communicator)
    return ucv, communicator


def release_ue_connection() -> None:
    """Drop UnrealCV TCP session. Use when reconnect fails; PIE restart is usually not needed."""
    global ucv, communicator
    g10k.release_connection(ucv, communicator=communicator)
    ucv = None
    communicator = None


print(f"[Paths] root={_root}")

[Paths] root=/home/winder17wsl_ishizawalab/01_Private/Program/SimWorld


In [ ]:
# Optional — 接続エラー時のみ。実行後 Cell 2 → Cell 5（PIE 再起動は通常不要）
# release_ue_connection()

[UE] session released


In [3]:
# マス番号 (gx, gy) — 1 始まり
HUMAN_CELL = (5, 5)
ROBOT_START = (10, 5)
ROBOT_GOAL = (50, 50)
GOAL_DWELL_S = 5.0

print(
    f"human={HUMAN_CELL}, robot {ROBOT_START} -> {ROBOT_GOAL}, dwell={GOAL_DWELL_S}s"
)

human=(5, 5), robot (10, 5) -> (50, 50), dwell=5.0s


In [4]:
# Re-run: Cell 5 only OK (uses patrol defaults if Cell 4 was not run).
_human = globals().get("HUMAN_CELL", patrol.HUMAN_CELL)
_robot_start = globals().get("ROBOT_START", patrol.ROBOT_START_CELL)
_robot_goal = globals().get("ROBOT_GOAL", patrol.ROBOT_GOAL_CELL)
_dwell = globals().get("GOAL_DWELL_S", patrol.GOAL_DWELL_S)
print(f"human={_human}, robot {_robot_start} -> {_robot_goal}, dwell={_dwell}s")

ucv, communicator = ensure_connection()
result = patrol.run_patrol_scenario(
    human_cell=_human,
    robot_start_cell=_robot_start,
    robot_goal_cell=_robot_goal,
    goal_dwell_s=_dwell,
    skip_perimeter_if_already_solid=True,
    skip_robot_probe=True,
    ucv=ucv,
    communicator=communicator,
)
result

human=(5, 5), robot (10, 5) -> (50, 50), dwell=5.0s
[UE] Probing UnrealCV on ['127.0.0.1', '10.103.0.1', '10.255.255.254'] (timeout=3s each) ...
[UE] probe OK: 127.0.0.1:9000


INFO:__init__:238:Got connection confirm: b'connected to SimWorld'


[UE] probe TCP failed 10.103.0.1:9000 — timed out
[UE] probe skip: 10.103.0.1:9000
[UE] probe TCP failed 10.255.255.254:9000 — [Errno 111] Connection refused
[UE] probe skip: 10.255.255.254:9000
=>Info: using ip-port socket
[UE] Connected via UnrealCV at 127.0.0.1:9000
[UE] Reusing notebook UnrealCV session at 127.0.0.1:9000
[Scenario] prepare PIE rerun (clear prior agents) ...
[Pak] OK pakchunk1000-Windows.pak: Mounted C:\SimWorldServer\SimWorld\Content\Paks\pakchunk1000-Windows.pak
[Pak] OK pakchunk0-Windows.pak: Mounted C:\SimWorldServer\SimWorld\Content\Paks\pakchunk0-Windows.pak
[Robot] skip BP probe (reuse session)
[Humanoid] skip BP probe (reuse session)
[Registry] scanning 10000 cube actors ...
[Registry] cache saved: /home/winder17wsl_ishizawalab/01_Private/Program/SimWorld/dev/grid_env_10k/.pie_block_registry.json
[Registry] mapped 396/396 blocks in 0.0s
[Scenario] skip perimeter (assume already T)
[Costmap] 30 m, resolution=0.30 m, lethal from 396 blocks, grid=(100, 100)
[UE

PatrolResult(perimeter_ok=True, human_name='GEN_BP_Humanoid_0', humanoid_on_floor=True, robot_spawned=True, outbound_arrived=True, return_arrived=True, start_xy=(285.0, 135.0), goal_xy=(1485.0, 1485.0), final_xy=(291.335, 158.182), return_dist_cm=24.032006761816607, humanoid_xy=(135.0, 135.0), humanoid_feet_z_cm=192.275)

In [5]:
success = patrol.patrol_success(result)
print(
    f"SUCCESS={success}, return_dist={result.return_dist_cm:.1f} cm, "
    f"humanoid_on_floor={result.humanoid_on_floor}, "
    f"humanoid_feet_z={result.humanoid_feet_z_cm}"
)

SUCCESS=True, return_dist=4.2 cm, humanoid_on_floor=True, humanoid_feet_z=150.0
